# Activity: Static Word Embedding & Large Word Embedding  

**Author:** Louis G. Binwag  
**References:** Course notebooks (`.ipynb`)  


# **Word Embeddings**

The requirements for this stage of the analysis are:

1. Most similar words (Using Static Word Embedding) from the top occurring words (max of 3 words, from Term-Frequency Language Model).
2. PCA of the words from the top 3 words (from the Term-Frequency Language Model).
3. Application of a Hugging Face model to your dataset (BERT-based topic model, BERT-based Sentiment / Emotion model)

As such, I am going to apply similarly the processes used in the lecture and course notes on the preprocessed dataset, considering mostly the `cleaned text` attribute of reddit_cleaned.csv

In [ ]:
!pip install scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
import joblib
import pandas as pd

from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [ ]:
corpus = pd.read_csv("../data/Reddit/reddit_clean.csv")
corpus.head()

## **Bag of Words Model**

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

added_stopwords = [
    "sa", "na", "ng", "ang", "mga", "yung", "pa", "lang", "naman", "ba",
    "ko", "mo", "nila", "pero", "daw", "din", "rin", "kasi", "nga", "eh",
    "si", "ni", "kay", "para", "kung", "ay", "lahat", "wala", "may", "meron",
    "dapat", "talaga", "ano", "oo", "hindi", "diba", "sobrang", "super",
    "mas", "isa", "dalawa", "tatlo", "sen", "say", "ma", "make", "ako", "ka"
]

custom_stopwords = list(set(ENGLISH_STOP_WORDS).union(added_stopwords))

In [ ]:
bow_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    # stop_words=custom_stopwords
)

doc2vec_bow = bow_vectorizer.fit_transform(corpus['post_full'])
doc2vec_bow = pd.DataFrame(doc2vec_bow.toarray(), columns=bow_vectorizer.get_feature_names_out())
doc2vec_bow.head()

# Save to file
doc2vec_bow.to_pickle('doc2vec_bow.pkl')

# Load from file
doc2vec_bow = pd.read_pickle('doc2vec_bow.pkl')

In [ ]:
def top_n_grams(document, top_n=10):
  return document.sort_values(ascending=False).index[:top_n].tolist()

In [ ]:
doc2vec_bow.apply(lambda document:
                  top_n_grams(document, top_n=5),
                  axis=1)

In [ ]:
# Wordcloud using BoW and TF-IDF
def plot_wordcloud(data, title):
  wordcloud = WordCloud(
    width=800, height=400, background_color='white'
  ).generate_from_frequencies(data)
  plt.figure(figsize=(10, 6))
  plt.imshow(wordcloud, interpolation='bilinear')
  plt.axis('off')
  plt.title(title)
  plt.show()

In [ ]:
bow_frequencies = doc2vec_bow.sum(axis=0).sort_values(ascending=False)
print(bow_frequencies.head(20))
plot_wordcloud(bow_frequencies, 'Wordcloud using BoW')

## **Analysis based on this BoW (Including knowledge from Data Preprocessing and EDA Steps)**

I still need to find the working way to implement stop-words. It seems that the dataset is still ignoring the stop-words i had implemented.

But for the sake of analysis, we can ignore the stopwords and the top words based on frequency would be:
1. Philippine
2. Agriculture
3. Rice
4. Farmer
5. Agricultural

Which could mean based on what I understand is that there are frequent conversations pertaining to Rice Farming. And based as well on my knowledge on the field, Rice-Farming is the most controversial part of agriculture, followed by Fisheries and then vegetations.

This could mean a number of possible things:

- Polarized discussions on Rice Farming
- Frequent concerns of farmers stemming from keywords found on the set
    - Undervalued (possibly as a profession or as a means)
    - Government or Political platform issues regarding farmers/farming/rice/agriculture
    - Land concerns, previously mentioned in EDA "Cynthia Villar" is reknowned for abusing farmers.

# **Implementing Static Word Embeddings**

In [ ]:
%pip install gensim

In [ ]:
import gensim

In [ ]:
corpus.head()

In [ ]:
# Tokenize the text
corpus['tokenized_text'] = corpus['cleaned_text'].apply(lambda x: x.split())

In [ ]:
display(
  corpus['cleaned_text'].head()
)

display(
  corpus['tokenized_text'].head()
)

In [ ]:
model = gensim.models.Word2Vec(corpus['tokenized_text'],
                               vector_size=100,  # Dimension of the word vectors
                               min_count=1,    # Ignores all words with total frequency lower min_count
                               sg=0,           # Training algorithm: 0 for CBOW; 1 for Skip-gram
                               window=5,       # Maximum distance between the current & predicted word within a sentence
                               workers=4,      # Number of workers to use for training
                               epochs=10       # Number of iterations over the corpus
                               )

model.save('word2vec.model')

In [ ]:
# Load the model
model = gensim.models.Word2Vec.load('word2vec.model')

# Get vectors from the model
vectors = model.wv

In [ ]:
# Number of word vectors generated (columns)
len(vectors)

In [ ]:
# List of words / tokens sorted by most frequent
vectors.index_to_key
vectors.index_to_key[:20]  

In [ ]:
# The similarity method returns a score of how similar the vectors of two words are. 
# The closer the score is to 1, the higher the similarity. The closer to 0, the lower the similarity.
vectors.similarity('agriculture', 'philippine')

In [ ]:

vectors.similarity('rice', 'philippine')

In [ ]:

vectors.similarity('farmer', 'philippine')

In [ ]:

vectors.similarity('rice', 'agriculture')

In [ ]:
# The most_similar method returns words whose vectors are most similar to the given word.
vectors.most_similar('farmer', topn=15)

In [ ]:
# The most_similar method returns words whose vectors are most similar to the given word.
vectors.most_similar('villar', topn=15)

In [ ]:
# Plot word vectors
def plot_vectors(vectors, words):
  # Create a figure and a 2D Axes
  fig, ax = plt.subplots()

  for word in words:
    # Get the vector for the word
    vector = vectors[word]

    # Plot the word near the arrow head
    ax.text(vector[0] + 0.01, vector[1] + 0.01, word, fontsize=9)
    # Plot the vector
    ax.arrow(
      0,                      # Start x
      0,                      # Start y
      vector[0],              # End x
      vector[1],              # End y
      # head_width=0.05,      # Arrow head width
      # head_length=0.1       # Arrow head length
      head_width=0.05 * 0.1,  # Arrow head width
      head_length=0.1 * 0.1,  # Arrow head length
      lw=0.01,                # Arrow line width
      # fc='r',                 # Arrow fill color
      # ec='r'                  # Arrow edge color
    )

  # Remove border around the plot
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.spines['left'].set_visible(False)

  plt.grid()

  # Display the plot
  plt.show()


plot_vectors(vectors, vectors.index_to_key[:10])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# Words most similar to "rice"
vectors.most_similar('rice', topn=5)
word_set1 = [word for word, score in vectors.most_similar('pogo', topn=5)]
word_set1

In [ ]:
# Words most similar to 'agriculture'
vectors.most_similar('agriculture', topn=5)
word_set2 = [word for word, score in vectors.most_similar('scam', topn=5)]
word_set2

In [ ]:
words_to_viz = word_set1 + word_set2
words_to_viz

In [ ]:
vecs_to_viz = [vectors[word] for word in words_to_viz]
vecs_to_viz

In [ ]:
pca = PCA(n_components=2)
vecs_PCA = pca.fit_transform(vecs_to_viz)
vecs_PCA

In [ ]:
vecs_PCA_df = pd.DataFrame(columns=['word', 'association', 'PC1', 'PC2'])

for idx in range(len(words_to_viz)):
  word = words_to_viz[idx]
  association = 'agriculture' if word in word_set1 else 'rice'
  PC1 = vecs_PCA[idx, 0]
  PC2 = vecs_PCA[idx, 1]
  new_row = [word, association, PC1, PC2]

  vecs_PCA_df.loc[len(vecs_PCA_df)] = new_row

vecs_PCA_df

In [ ]:
vecs_viz = sns.scatterplot(data=vecs_PCA_df, x='PC1', y='PC2', hue='association')
for idx in range(len(words_to_viz)):
  vecs_viz.text(vecs_PCA_df.PC1[idx] - 0.01, vecs_PCA_df.PC2[idx], vecs_PCA_df.word[idx], horizontalalignment='right',
                size='x-small', color='black', weight='light')
vecs_viz

plt.show()

In [ ]:
def viz_associated_wordvecs(word1, word2, topn=5, model=vectors):
  word_set1 = [model.most_similar(word1, topn=topn)[idx][0] for idx in range(topn)]
  word_set2 = [model.most_similar(word2, topn=topn)[idx][0] for idx in range(topn)]
  words_to_viz = word_set1 + word_set2
  vecs_to_viz = [model[word] for word in words_to_viz]

  pca = PCA(n_components=2)
  vecs_PCA = pca.fit_transform(vecs_to_viz)
  vecs_PCA_df = pd.DataFrame(columns=['word', 'association', 'PC1', 'PC2'])

  for idx in range(len(words_to_viz)):
    word = words_to_viz[idx]
    association = word1 if word in word_set1 else word2
    PC1 = vecs_PCA[idx, 0]
    PC2 = vecs_PCA[idx, 1]
    new_row = [word, association, PC1, PC2]
    vecs_PCA_df.loc[len(vecs_PCA_df)] = new_row

  vecs_viz = sns.scatterplot(data=vecs_PCA_df, x='PC1', y='PC2', hue='association')

  for idx in range(len(words_to_viz)):
    vecs_viz.text(vecs_PCA_df.PC1[idx] - 0.01, vecs_PCA_df.PC2[idx], vecs_PCA_df.word[idx], horizontalalignment='right',
                  size='x-small', color='black', weight='light')
  return vecs_PCA_df

In [ ]:
viz_associated_wordvecs('rice', 'farm', topn=5, model=vectors)

In [ ]:
viz_associated_wordvecs('philippine', 'farm', topn=5, model=vectors)

In [ ]:
viz_associated_wordvecs('villar', 'farm', topn=5, model=vectors)

In [ ]:
viz_associated_wordvecs('profession', 'farmer', topn=5, model=vectors)

## **Short Analysis for Static Word Embeddings and PCA**

Looking at these graphical representation (PCA) of Static Word Embeddings, it looks like Agriculture and Philippines does not seem to be aligned.

Despite being geographically advantageous as a agricultural country, based on these embeddings it seems that a lot of sentiments regarding agriculture is the fact that agriculture is undervalued. Historically, this probably stems from the lack of governmental support for our agriculture, or, could be an on-going trend especially that 'tech' or 'med' is becoming the hit profession, which causes stereotyping when it comes to agriculture or farming, making it undervalued.

However, this could also stem from the fact that the dataset may be heavily populated with tagalog words, and the methods applied may be inefficient (could probably require fine-tuning for tagalog-lexicon?)

# **Large Word Embeddings**

In [ ]:
%pip install transformers triton

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from transformers import pipeline
from tqdm import tqdm
tqdm.pandas()

In [ ]:
corpus[["created_utc","post_title","cleaned_text"]].head()

In [ ]:
# from transformers import pipeline

# # https://huggingface.co/tabularisai/multilingual-sentiment-analysis

# # @misc{tabularisai_2025,
# #     author       = { tabularisai and Samuel Gyamfi and Vadim Borisov and Richard H. Schreiber },
# #     title        = { multilingual-sentiment-analysis (Revision 69afb83) },
# #     year         = 2025,
# #     url          = { https://huggingface.co/tabularisai/multilingual-sentiment-analysis },
# #     doi          = { 10.57967/hf/5968 },
# #     publisher    = { Hugging Face }
# # }

# # Load the classification pipeline with the specified model
# pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

# # Classify a new sentence
# sentence = "I love this product! It's amazing and works perfectly."
# result = pipe(sentence)

# # Print the result
# print(result)

sentiment_analyzer = pipeline('sentiment-analysis',
                              model='nlptown/bert-base-multilingual-uncased-sentiment',
                              device=0)

In [ ]:
result = sentiment_analyzer(
  "Hugging Face's BERT models are great."
)
print(result)

In [ ]:
short_title = corpus[corpus['post_title'].str.len() < 512].copy()
short_title.tail()

In [ ]:
sentiment_analyzer(short_title["post_title"].iloc[0])


In [ ]:
short_title[['sentiment', 'sentiment_confidence']] = short_title.progress_apply(
  lambda row: pd.Series(sentiment_analyzer(row['post_title'])[0]),
  axis=1
)

In [ ]:
short_title.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))
short_title['sentiment_confidence'].hist(ax=ax)

# Set labels
ax.set_xlabel('Sentiment Confidence')
ax.set_ylabel('Frequency')

# 45 degree angle for x-axis labels
ax.tick_params(axis='x', rotation=45)

# No spines
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

short_title['sentiment'].value_counts()

In [ ]:
# Histogram of sentiment confidence along with the sentiment
fig, ax = plt.subplots(5, figsize=(30, 10))

short_title['sentiment_confidence'].hist(
  by=short_title['sentiment'],
  ax=ax,
)

# Add more space between subplots
plt.subplots_adjust(hspace=0.9)

# Set labels
ax[4].set_xlabel('Sentiment Confidence')
for a in ax:
  a.set_ylabel('Frequency')
  # 45 degree angle for x-axis labels
  a.tick_params(axis='x', rotation=45)

plt.show()

In [ ]:
%pip install bertopic==0.16.0 transformers==4.44.2


In [ ]:
import bertopic, transformers
print("BERTopic:", bertopic.__version__)
print("Transformers:", transformers.__version__)


In [ ]:
from bertopic import BERTopic

In [ ]:
# fill NaN values with empty strings
corpus['cleaned_text'] = corpus['cleaned_text'].fillna('blank')

In [ ]:
topic_model = BERTopic(language="multilingual",
                       calculate_probabilities=True, verbose=True,
                       n_gram_range=(1, 1), min_topic_size=3
                       )
topics, probs = topic_model.fit_transform(corpus['cleaned_text'])

In [ ]:
len(corpus)

In [ ]:
topic_dataframe = topic_model.get_topic_info()
topic_dataframe

In [ ]:
topic_model.get_topic(-1)

In [ ]:
topic_model.topics_

In [ ]:
try:
  topic_model.visualize_topics()
except ValueError:
  # Dataset too small
  print("Dataset too small to visualize topics")

In [ ]:
topic_model.visualize_distribution(probs[10], min_probability=0.015)

## **Short Analysis on BERT-Based Topic Modelling**

Consistent with the previous outputs, Agriculture and Politics are heavily mixed. As I thought about it, agriculture lacks support from the government / politics of the country, despite it being consistently called out periodically. 

In this case, it concerns the more recent political issues considering the Blue Ribbon Committee where they discuss on-going issues and much needed investigations on various sectors of the country. 

However, if my thought process is wrong on the topic of at least how this pipeline is worked out, it seems to me that a lot of the topic or post titles focus on politics and agriculture.



In [ ]:
toxic_analyzer = pipeline('text-classification',
                          model='unitary/toxic-bert',
                          device=0)

In [ ]:
result = toxic_analyzer(
  short_title.iloc[9]['post_title']
)
print(result)
print(short_title.iloc[9]['post_title'])

In [ ]:
short_title[['toxicity', 'toxicity_confidence']] = short_title.progress_apply(
  lambda row: pd.Series(toxic_analyzer(row['post_title'])[0]),
  axis=1
)

In [ ]:
short_title

## **My Next Steps Moving Forward**

as per our previous conversation:

```
A small update would be what agricultural legislative and/or executive agenda should be put forward based on topics and sentiments/emotions. Imagine, if we can process that the main problem area is on micro-loans, or agricultural insurance.

This can be mined and extracted with how intense the emotion/sentiment to that particular agenda.
```

I plan to try to extract main problem area relating to agriculture rather than just the *generalized fact* that the government does not support agriculture as well as other neighbouring agricultural countries (like Thailand, Vietnam, who've all surpassed us the last 60 years). I think this scope is interesting as it helps me visualize through data what are the underlying sentiments that come from the topic ***Agriculture***

**There are quite a few changes I might go through:**
- For a faster process, I opted to simply use `post_title` (and a cleaned version of it) since I believe this already encompass sentiments and summarizes the content of the post. However, seeing these results, it ***may*** be beneficial in the long run.
    - This is also in conjunction to the fact that I had a hard time managing my deadlines due to sudden changes in class schedules of other subjects. However, I will continue to work on this as it is also beneficial to my knowledge on Social Computing (The area of our thesis)
- Considering sentiment analysis pertains to text-based content, I might consider **concatinating** `post_title` and `comments` and cleaning them. Then, feeding it through the whole pipeline again, and see if there is any difference. 
- I think I could also **include** `post_body` since this is commonly the expounding of `post_title` whilst `post_title` is the summary of `post_body`
 